In [1]:
import ee
import geemap
ee.Authenticate()
ee.Initialize()

In [2]:
Map = geemap.Map()

# Panama boundary
countries = ee.FeatureCollection("FAO/GAUL/2015/level0")
panama_fc = countries.filter(ee.Filter.eq("ADM0_NAME", "Panama"))
panama_geom = panama_fc.geometry()

Map.centerObject(panama_geom, 7)

In [3]:
districts = ee.FeatureCollection("projects/deforestation-panama/assets/Districts")

In [4]:
health_centers = ee.FeatureCollection("projects/deforestation-panama/assets/health_institutions")

In [6]:
print(health_centers.first().propertyNames().getInfo())

['TIPO', 'DISTRITO', 'CORREG', 'PROV', 'NOMBRE', 'system:index', 'BARRIO', 'GlobalID']


In [8]:
health_simple = health_centers.map(
    lambda f: ee.Feature(None, {
        'HEALTH_INSTITUTION': f.get('NOMBRE'),
        'DISTRICT': f.get('DISTRITO')
    })
)

In [9]:
import pandas as pd

features_info = health_simple.getInfo()['features']

rows = []

for f in features_info:
    props = f['properties']

    row = {
        'HEALTH_INSTITUTION': props.get('HEALTH_INSTITUTION'),
        'DISTRICT': props.get('DISTRICT')
    }

    rows.append(row)

df = pd.DataFrame(rows)

df = df.sort_values(
    by=['DISTRICT', 'HEALTH_INSTITUTION']
).reset_index(drop=True)

print(df)

        HEALTH_INSTITUTION   DISTRICT
0                                    
1                                    
2          Centro de Salud  Aguadulce
3          Centro de Salud  Aguadulce
4          Centro de Salud  Aguadulce
...                    ...        ...
1227  Centro Médico Tonosí     Tonosí
1228       Centro de Salud     Tonosí
1229    Farmacia San Roque     Tonosí
1230       Farmacia Tonosí     Tonosí
1231       Hospital Tonosí     Tonosí

[1232 rows x 2 columns]


In [10]:
health_counts = (
    df.groupby('DISTRICT')
      .size()
      .reset_index(name='NUMBER_OF_HEALTH_INSTITUTIONS')
)

print(health_counts)

       DISTRICT  NUMBER_OF_HEALTH_INSTITUTIONS
0                                            2
1     Aguadulce                             22
2        Alanje                              3
3         Antón                              6
4      Arraiján                             29
..          ...                            ...
60  Santa María                              2
61     Santiago                             49
62         Soná                             10
63         Tolé                              2
64       Tonosí                              5

[65 rows x 2 columns]


In [ ]:
df.to_csv('Panama_Health_Institutions_by_District.csv', index=False)

health_counts.to_csv('Panama_Health_Institution_Counts.csv', index=False)